# Plantilla de pronóstico

Abordajes diarios en Chicago (CTA). La carga de datos y el dataset base ya están listos. Completa la subclase del dataset y el modelo según la tarea que elijas:

| Tarea | Objetivo del dataset | Salida del modelo |
| --- | --- | --- |
| Día siguiente | tren del día que sigue a la ventana | `(batch, 1)`, desde el último paso |
| Vector de 14 días | los próximos 14 días de tren | `(batch, 14)`, desde el último paso |
| Seq2seq | en cada día de la ventana, los 14 días de tren que siguen | `(batch, 56, 14)`, RNN, GRU o WaveNet |

Bloques por completar, marcados con `TODO`:

1. `TimeSeriesDataset_multivariate`
2. argumentos del dataset (`output_size` si el horizonte es de varios días)
3. el modelo
4. en seq2seq, el paso que se mide y se grafica

## 1. Carga de datos

Leemos el CSV de abordajes diarios, acortamos los nombres de las columnas y usamos la fecha como índice. Quitamos `total` (es la suma de bus y tren) y las filas duplicadas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device

In [ ]:
path = Path('CTA_-_Ridership_-_Daily_Boarding_Totals.csv')
df = pd.read_csv(path, parse_dates=["service_date"], thousands=",")
df.columns = ["date", "day_type", "bus", "rail", "total"]  # nombres cortos
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # total = bus + tren
df = df.drop_duplicates()  # meses duplicados: 2011-10 y 2014-07
df.head()

## 2. Series multivariadas

La entrada usa tren y bus, en millones de abordajes, y el tipo de día siguiente en one-hot. Ese tipo se conoce de antemano.

In [ ]:
df_mulvar = df[["rail", "bus"]] / 1e6  # use both rail & bus series as input
df_mulvar["next_day_type"] = df["day_type"].shift(-1)  # we know tomorrow's type
df_mulvar = pd.get_dummies(df_mulvar, dtype=float)  # one-hot encode day type
df_mulvar.head()

## 3. Series a tensores

Cada fila pasa a un tensor `float32` con las cinco columnas. Entrenamiento: 2016 a 2018. Validación: enero a mayo de 2019. Prueba: desde junio de 2019.

In [ ]:
train = torch.tensor(df_mulvar["2016-01-01":"2018-12-31"].values, dtype=torch.float32)
valid = torch.tensor(df_mulvar["2019-01-01":"2019-05-31"].values, dtype=torch.float32)
test = torch.tensor(df_mulvar["2019-06-01":].values, dtype=torch.float32)
train.shape, valid.shape, test.shape

## 4. Dataset con ventanas y DataLoaders

`TimeSeriesDataset` arma pares de una ventana y el valor que sigue. La subclase `TimeSeriesDataset_multivariate` cambia ese objetivo según la tarea. Los DataLoaders entregan los pares en lotes de 32. El de entrenamiento se baraja; la validación conserva el orden.

In [ ]:
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series, window_length):
        self.series = series
        self.window_length = window_length

    def __len__(self):
        return len(self.series) - self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length  # 1st index after window
        window = self.series[idx : end]
        target = self.series[end]
        return window, target

Completa la subclase. La ventana siempre es `self.series[idx : idx + window_length]`, de forma `(56, 5)`.

- **Día siguiente.** El objetivo es el tren (columna 0) del día que sigue a la ventana. Puedes partir de `super().__getitem__`.
- **Vector de 14 días.** El objetivo es `self.series[end : end + output_size, 0]`. Ajusta `__len__` para que ese tramo quepa en la serie: `len(self.series) - window_length - output_size + 1`.
- **Seq2seq.** El mismo ajuste de `__len__`. El objetivo tiene un vector de `output_size` días de tren por cada día de la ventana. `unfold` arma esos tramos solapados, con forma `(window_length, output_size)`.

In [ ]:
class TimeSeriesDataset_multivariate(TimeSeriesDataset):
    def __init__(self, series, window_length, output_size=1):
        super().__init__(series, window_length)
        self.output_size = output_size

    def __len__(self):
        # TODO: si el objetivo se extiende después de la ventana, acorta el dataset.
        return super().__len__()

    def __getitem__(self, idx):
        # TODO: devuelve (ventana, objetivo) con la forma de tu tarea.
        # Día siguiente: objetivo con el tren del día que sigue.
        # Vector: objetivo con los próximos output_size días de tren.
        # Seq2seq: objetivo (window_length, output_size), un horizonte por día.
        raise NotImplementedError

In [ ]:
WINDOW = 56

# TODO: agrega output_size si tu subclase predice varios días.
# dataset_kwargs = dict(window_length=WINDOW, output_size=14)
dataset_kwargs = dict(window_length=WINDOW)

train_set = TimeSeriesDataset_multivariate(train, **dataset_kwargs)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)
valid_set = TimeSeriesDataset_multivariate(valid, **dataset_kwargs)
valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=32, shuffle=False)

window, target = next(iter(train_loader))
window.shape, target.shape

## 5. Modelo

Completa `ForecastModel`. La entrada `x` tiene forma `(batch, ventana, 5)`.

- **RNN o GRU de un solo pronóstico.** `batch_first=True`. La capa lineal lee `output[:, -1]` y devuelve `(batch, output_size)`.
- **RNN o GRU seq2seq.** La misma recurrente. La lineal se aplica a cada paso y devuelve `(batch, ventana, output_size)`.
- **WaveNet.** Convoluciones causales (padding solo a la izquierda) y una lineal al final. Permuta para que el canal quede en la dimensión que espera `Conv1d`, y vuelve a dejar el tiempo en el medio. La salida es `(batch, ventana, output_size)`.

`input_size` es 5. `output_size` es 1 para el día siguiente y 14 para el horizonte de dos semanas.

In [ ]:
class ForecastModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # TODO: define las capas del modelo que elegiste.
        self.output_size = output_size

    def forward(self, x):
        # TODO: devuelve el pronóstico.
        # Un solo pronóstico: (batch, output_size).
        # Seq2seq o WaveNet: (batch, ventana, output_size).
        raise NotImplementedError


# TODO: fija output_size en 1 o en 14.
model = ForecastModel(input_size=5, hidden_size=32, output_size=None).to(device)
model

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(window.to(device)).cpu()
pred.shape, target.shape

## 6. Entrenamiento

Pérdida de Huber y Adam (`lr=0.001`). Cada 50 épocas se mide el MAPE en validación.

En seq2seq la pérdida usa todos los pasos. El MAPE de validación usa solo el pronóstico del último día de la ventana: descomenta esas dos líneas.

In [ ]:
import torchmetrics

loss_fn = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
MAPE = torchmetrics.MeanAbsolutePercentageError().to(device)
n_epochs = 500  # bájalo mientras compruebas que las formas coinciden

In [ ]:
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    MAPE.reset()
    for batch in train_loader:
        window, target = batch
        window, target = window.to(device), target.to(device)
        pred = model(window)
        loss = loss_fn(pred, target)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    if epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            for window, target in valid_loader:
                window = window.to(device)
                target = target.to(device)
                pred = model(window)
                # TODO: seq2seq. Descomenta para medir solo el último día.
                # pred = pred[:, -1]
                # target = target[:, -1]
                MAPE.update(pred, target)
        mean_loss = total_loss / len(train_loader)
        mape = MAPE.compute()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}, MAPE: {mape:.4f}")

## 7. Pronóstico en el conjunto de prueba

La primera ventana de prueba produce el pronóstico. La figura muestra esa ventana de tren, el abordaje real y el pronóstico. En seq2seq, grafica el pronóstico del último día de la ventana.

In [ ]:
import matplotlib.pyplot as plt

test_set = TimeSeriesDataset_multivariate(test, **dataset_kwargs)
window, target = test_set[0]
model.eval()
with torch.no_grad():
    pred = model(window.unsqueeze(0).to(device)).cpu().squeeze(0)

# TODO: seq2seq. Descomenta para graficar solo el último día.
# pred = pred[-1]
# target = target[-1]

forecast = pred.numpy()
actual = target.numpy()
horizon = len(np.atleast_1d(forecast))

dates = df_mulvar.loc["2019-06-01":].index[: WINDOW + horizon]
history = window[:, 0].numpy()
plt.plot(dates[:WINDOW], history, label="entrada")
plt.plot(dates[WINDOW - 1:], np.r_[history[-1], actual], label="real")
plt.plot(dates[WINDOW - 1:], np.r_[history[-1], forecast], ":", label="pronóstico")
plt.ylabel("Tren (millones)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()